In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import nest_asyncio
nest_asyncio.apply()

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
pio.renderers.default = "vscode"

import matplotlib.pyplot as plt
import matplotlib.pylab as pylab
import matplotlib.dates as mdates
plt.style.use('ggplot')
params = {'legend.fontsize': 'x-large',
        'figure.figsize': (16, 9),
        'axes.labelsize': 'x-large',
        'axes.titlesize':'x-large',
        'xtick.labelsize':'x-large',
        'ytick.labelsize':'x-large'}
pylab.rcParams.update(params)

import pandas as pd
import numpy as np
import QuantLib as ql
import datetime
import pytz
NY_tz = pytz.timezone("America/New_York")
CHI_tz = pytz.timezone("America/Chicago")
UTC_tz = pytz.timezone("UTC")

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
import sys
sys.path.append("../")

from BT.misc import ql_cal_date_range
from MDP.IRSwaps.IRSwapsMDP import IRSwapsMDP
from TB.IRSwapsTB import IRSwapsTB

from Query.IRSwaps.IRSwapQuery import IRSwapQuery
from Query.IRSwaps.IRSwapStructure import IRSwapStructure
from Query.IRSwaps.IRSwapValue import IRSwapValue

from BT.signals.pca_momentum import (
    PCAMomentumConfig,
    generate_ma_features,
    generate_labels,
    fit_pca_momentum,
    predict_signals,
    extract_pca_components,
)
from BT.signals.vectorized_backtest import vectorized_backtest, grid_search

# Configuration

Adjust curve source, time range, intraday frequency, and contract universe below.

In [ ]:
# ========== CURVE SOURCE ==========
CURVE_SOURCE = "BARCHART_STIRF-RL"
CURVE_NAME = "USD-SOFR-1D-Q12STIRT"
CURVE_ID = "USD-SOFR-1D"

# ========== TIME RANGE & FREQUENCY ==========
START = CHI_tz.localize(datetime.datetime(2025, 10, 1, 7, 0))
END = CHI_tz.localize(datetime.datetime(2026, 3, 12, 15, 0))
FREQ = "30min"  # intraday frequency: "15min", "30min", "1h"

# ========== CONTRACT UNIVERSE ==========
# Quarterly SFR outrights
SFR_OUTRIGHTS = [
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_1xIMM_2", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR1
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_2xIMM_3", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR2
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_3xIMM_4", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR3
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_4xIMM_5", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR4
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_5xIMM_6", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR5
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_6xIMM_7", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR6
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_7xIMM_8", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR7
    IRSwapQuery(curve=CURVE_ID, tenor="IMM_8xIMM_9", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),   # SFR8
]

# FOMC outrights
FOMC_OUTRIGHTS = [
    IRSwapQuery(curve=CURVE_ID, tenor="fomc_1", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, tenor="fomc_2", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, tenor="fomc_3", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, tenor="fomc_4", structure=IRSwapStructure.OUTRIGHT, structure_kwargs={"bpv": 1}),
]

# Curve spreads
SFR_CURVES = [
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.CURVE, name="SFR1/SFR2",
                structure_kwargs={"front_tenor": "IMM_1xIMM_2", "back_tenor": "IMM_2xIMM_3", "bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.CURVE, name="SFR2/SFR3",
                structure_kwargs={"front_tenor": "IMM_2xIMM_3", "back_tenor": "IMM_3xIMM_4", "bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.CURVE, name="SFR3/SFR4",
                structure_kwargs={"front_tenor": "IMM_3xIMM_4", "back_tenor": "IMM_4xIMM_5", "bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.CURVE, name="FOMC1/FOMC2",
                structure_kwargs={"front_tenor": "fomc_1", "back_tenor": "fomc_2", "bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.CURVE, name="FOMC2/FOMC3",
                structure_kwargs={"front_tenor": "fomc_2", "back_tenor": "fomc_3", "bpv": 1}),
]

# Butterfly spreads
SFR_FLIES = [
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.FLY, name="SFR1/SFR2/SFR3 FLY",
                structure_kwargs={"front_tenor": "IMM_1xIMM_2", "belly_tenor": "IMM_2xIMM_3", "back_tenor": "IMM_3xIMM_4", "bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.FLY, name="SFR2/SFR3/SFR4 FLY",
                structure_kwargs={"front_tenor": "IMM_2xIMM_3", "belly_tenor": "IMM_3xIMM_4", "back_tenor": "IMM_4xIMM_5", "bpv": 1}),
    IRSwapQuery(curve=CURVE_ID, structure=IRSwapStructure.FLY, name="FOMC1/FOMC2/FOMC3 FLY",
                structure_kwargs={"front_tenor": "fomc_1", "belly_tenor": "fomc_2", "back_tenor": "fomc_3", "bpv": 1}),
]

ALL_QUERIES = SFR_OUTRIGHTS + FOMC_OUTRIGHTS + SFR_CURVES + SFR_FLIES
print(f"Total contracts in universe: {len(ALL_QUERIES)}")

# Data Fetching

Build intraday time grid and fetch rate time series via `IRSwapsTB.get_timeseries(timestamps=...)`.

**Note:** First run is slow (curve builds + API calls). Subsequent runs hit DiskCache.

In [ ]:
# Build business-day-filtered intraday time grid
cal = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
ts_range = ql_cal_date_range(
    cal, start=START, end=END, freq=FREQ,
    open_time=datetime.time(7, 0),
    close_time=datetime.time(15, 0),
)
print(f"Time grid: {len(ts_range)} timestamps from {ts_range[0]} to {ts_range[-1]}")

# Fetch via IRSwapsTB (DiskCache-backed)
curve_mdp = IRSwapsMDP(source=CURVE_SOURCE)
tb = IRSwapsTB(mdp=curve_mdp)

rate_df = tb.get_timeseries(
    start=None,
    end=None,
    timestamps=ts_range,
    queries=ALL_QUERIES,
    n_jobs=1,
)

print(f"Rate DataFrame: {rate_df.shape}")
display(rate_df.head(10))
display(rate_df.describe())

# Rate Visualization

In [ ]:
n_outright = len(SFR_OUTRIGHTS) + len(FOMC_OUTRIGHTS)
outright_cols = [c for c in rate_df.columns[:n_outright] if c in rate_df.columns]
struct_cols = [c for c in rate_df.columns[n_outright:] if c in rate_df.columns]

fig, axes = plt.subplots(2, 1, figsize=(16, 12), sharex=True)

for col in outright_cols:
    axes[0].plot(rate_df.index, rate_df[col], linewidth=0.6, label=col)
axes[0].legend(fontsize='small', ncol=3)
axes[0].set_title("SOFR STIR Outright Rates (Intraday)")
axes[0].set_ylabel("Rate (%)")

for col in struct_cols:
    axes[1].plot(rate_df.index, rate_df[col], linewidth=0.6, label=col)
axes[1].legend(fontsize='small', ncol=3)
axes[1].set_title("SOFR STIR Curve / Fly Rates (Intraday)")
axes[1].set_ylabel("Rate (%)")

plt.tight_layout()
plt.show()

# Single Contract PCA Momentum Signal

Pick one contract for detailed analysis: feature generation, PCA decomposition, and signal prediction.

In [ ]:
# Pick target contract
TARGET_COL = rate_df.columns[0]  # e.g. SFR1 outright
target_series = rate_df[TARGET_COL].dropna()
print(f"Target: {TARGET_COL} | {len(target_series)} observations")

# Configure PCA momentum
config = PCAMomentumConfig(
    ewma_short_spans=[24, 48, 120, 240],
    ewma_long_spans=[360, 1080, 1440],
    n_components=2,
    classifier="logistic",
    classifier_C=1.0,
    label_z_threshold=1.0,
    label_lookback=120,
    label_forward_window=10,
    train_fraction=0.7,
)

# Generate features, labels, fit, predict
features = generate_ma_features(target_series, config.ewma_short_spans, config.ewma_long_spans)
labels = generate_labels(target_series, config.label_forward_window, config.label_lookback, config.label_z_threshold)
pipeline, train_idx = fit_pca_momentum(features, labels, config)
signals = predict_signals(pipeline, features, smoothing=config.signal_smoothing)
pc_components = extract_pca_components(pipeline, features)

print(f"\nFeatures shape: {features.shape}")
print(f"PCA explained variance: {pipeline.named_steps['pca'].explained_variance_ratio_}")
print(f"Total variance explained: {sum(pipeline.named_steps['pca'].explained_variance_ratio_):.3f}")
print(f"\nLabel distribution (all):\n{labels.value_counts().sort_index()}")
print(f"\nSignal distribution:\n{signals.value_counts().sort_index()}")

# PCA Component Visualization

- PC1 = Master Trend (consensus momentum across all MA horizons)
- PC2 = Acceleration / Mean-Reversion (short vs long MA divergence)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# PC1 vs PC2 scatter colored by label
valid_pc = pc_components.dropna()
scatter_labels = labels.reindex(valid_pc.index).fillna(0)
colors = {-1.0: 'red', 0.0: 'gray', 1.0: 'blue'}
for lbl, color in colors.items():
    mask = scatter_labels == lbl
    axes[0, 0].scatter(
        valid_pc.loc[mask, "PC1"], valid_pc.loc[mask, "PC2"],
        c=color, alpha=0.3, s=3, label=f"Label={int(lbl)}"
    )
axes[0, 0].set_xlabel("PC1 (Master Trend)")
axes[0, 0].set_ylabel("PC2 (Acceleration)")
axes[0, 0].legend()
axes[0, 0].set_title("PCA Space")

# PC1 time series
axes[0, 1].plot(valid_pc.index, valid_pc["PC1"], linewidth=0.5)
axes[0, 1].set_title("PC1 (Master Trend) Over Time")
axes[0, 1].axhline(0, color='black', linewidth=0.5, linestyle='--')

# PC2 time series
axes[1, 0].plot(valid_pc.index, valid_pc["PC2"], linewidth=0.5, color='orange')
axes[1, 0].set_title("PC2 (Acceleration) Over Time")
axes[1, 0].axhline(0, color='black', linewidth=0.5, linestyle='--')

# Signal overlay on rate
axes[1, 1].plot(target_series.index, target_series, linewidth=0.5, alpha=0.7, color='black', label="Rate")
buy = signals == 1
sell = signals == -1
axes[1, 1].scatter(target_series.index[buy.reindex(target_series.index, fill_value=False)],
                   target_series[buy.reindex(target_series.index, fill_value=False)],
                   c='blue', s=2, alpha=0.3, label="Long")
axes[1, 1].scatter(target_series.index[sell.reindex(target_series.index, fill_value=False)],
                   target_series[sell.reindex(target_series.index, fill_value=False)],
                   c='red', s=2, alpha=0.3, label="Short")
axes[1, 1].legend()
axes[1, 1].set_title(f"Signal Overlay: {TARGET_COL}")

plt.tight_layout()
plt.show()

# Single Vectorized Backtest

In [ ]:
# Annualization factor: 252 trading days * (8h session / 0.5h per bar) = 252 * 16
FREQ_TO_ANNUAL = {
    "5min": 252 * 8 * 12,
    "10min": 252 * 8 * 6,
    "15min": 252 * 8 * 4,
    "30min": 252 * 8 * 2,
    "1h": 252 * 8,
}
annual_factor = FREQ_TO_ANNUAL.get(FREQ, 252 * 8 * 2)

result = vectorized_backtest(
    target_series, signals,
    bpv=1.0,
    tc_bps=0.25,
    annualization_factor=annual_factor,
)

print(f"{'='*40}")
print(f"Contract:     {TARGET_COL}")
print(f"Total Return: {result.total_return:.2f}")
print(f"Sharpe Ratio: {result.sharpe:.3f}")
print(f"Max Drawdown: {result.max_drawdown:.2f}")
print(f"Num Trades:   {result.n_trades}")
print(f"Win Rate:     {result.win_rate:.1%}")
print(f"Avg Hold:     {result.avg_holding_periods:.1f} periods")
print(f"{'='*40}")

fig, axes = plt.subplots(2, 1, figsize=(16, 10), sharex=True)

axes[0].plot(result.pnl_series.index, result.pnl_series.values, linewidth=1)
axes[0].set_title(f"Cumulative P&L: {TARGET_COL} | Sharpe={result.sharpe:.2f}")
axes[0].set_ylabel("Cum P&L (bps * bpv)")
axes[0].axhline(0, color='black', linewidth=0.5)

running_max = result.pnl_series.cummax()
dd = result.pnl_series - running_max
axes[1].fill_between(dd.index, dd.values, 0, alpha=0.5, color='red')
axes[1].set_title("Drawdown")
axes[1].set_ylabel("Drawdown")

plt.tight_layout()
plt.show()

# Grid Search

Search over PCA momentum hyperparameters **and** contract selection simultaneously.

Dimensions:
- MA span configurations
- PCA n_components
- Classifier regularization (C)
- Label z-score threshold (dead-zone width)
- Forward return window
- Contract (which SFR/FOMC product)

In [ ]:
param_grid = {
    "ewma_short_spans": [
        [24, 48, 120, 240],       # default hourly-scaled
        [12, 24, 48, 96],         # faster (more sensitive to short-term moves)
        [48, 120, 240, 480],      # slower (less whipsaw)
    ],
    "ewma_long_spans": [
        [360, 1080, 1440],        # default hourly-scaled
        [240, 480, 960],          # shorter macro horizon
    ],
    "n_components": [2, 3],
    "classifier_C": [0.1, 1.0, 10.0],
    "label_z_threshold": [0.5, 1.0, 1.5],
    "label_forward_window": [5, 10, 20],
}

results_df = grid_search(
    rate_df=rate_df,
    param_grid=param_grid,
    contracts=None,  # search all columns
    bpv=1.0,
    tc_bps=0.25,
    annualization_factor=annual_factor,
    min_observations=500,
    show_tqdm=True,
)

results_df = results_df.dropna(subset=["sharpe"]).sort_values("sharpe", ascending=False)
print(f"Grid search complete: {len(results_df)} valid combinations")
display(results_df.head(20))

# Grid Search Results Analysis

In [ ]:
# ========== Best Sharpe per Contract ==========
best_per_contract = results_df.groupby("contract")["sharpe"].max().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, max(6, len(best_per_contract) * 0.4)))
best_per_contract.plot(kind='barh', ax=ax, color='steelblue')
ax.set_xlabel("Best Sharpe Ratio")
ax.set_title("Best Sharpe by Contract (Grid Search)")
ax.axvline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()

# ========== Heatmap: Sharpe by C vs z_threshold (best contract) ==========
best_contract = best_per_contract.idxmax()
subset = results_df[results_df["contract"] == best_contract]

if "classifier_C" in subset.columns and "label_z_threshold" in subset.columns:
    pivot = subset.pivot_table(
        index="classifier_C", columns="label_z_threshold",
        values="sharpe", aggfunc="max"
    )
    fig, ax = plt.subplots(figsize=(10, 6))
    im = ax.imshow(pivot.values, aspect='auto', cmap='RdYlGn', interpolation='nearest')
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([f"{v:.1f}" for v in pivot.columns])
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.1f}" for v in pivot.index])
    ax.set_xlabel("label_z_threshold")
    ax.set_ylabel("classifier_C")
    ax.set_title(f"Sharpe Heatmap: {best_contract}")
    for i in range(len(pivot.index)):
        for j in range(len(pivot.columns)):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9)
    plt.colorbar(im)
    plt.tight_layout()
    plt.show()

# ========== Top 10 parameter combos ==========
print("\nTop 10 Parameter Combinations:")
display(results_df.head(10)[["contract", "sharpe", "total_return", "max_drawdown", "n_trades", "win_rate",
                              "n_components", "classifier_C", "label_z_threshold", "label_forward_window"]])

# Production Integration: QueryDrivenBacktest

Demonstrates how to plug the winning PCA momentum signal into the event-driven `QueryDrivenBacktest` for production-fidelity simulation with proper position handling and MTM.

In [ ]:
import operator
from BT.data_handler import TimeGrid
from BT.query_actions import AddQueryAction, UnwindPositionsAction
from BT.query_engine import QueryDrivenBacktest
from BT.query_strategy import QueryStrategy
from BT.triggers import Trigger, MktTriggerRequirements, AggregateTriggerRequirements

# Use the best parameters from grid search
best_row = results_df.iloc[0]
print(f"Best params: {best_row.to_dict()}")

# Rebuild signal with best params
best_contract_col = best_row["contract"]
best_series = rate_df[best_contract_col].dropna()

best_config = PCAMomentumConfig(
    ewma_short_spans=best_row.get("ewma_short_spans", [24, 48, 120, 240]),
    ewma_long_spans=best_row.get("ewma_long_spans", [360, 1080, 1440]),
    n_components=int(best_row.get("n_components", 2)),
    classifier="logistic",
    classifier_C=float(best_row.get("classifier_C", 1.0)),
    label_z_threshold=float(best_row.get("label_z_threshold", 1.0)),
    label_forward_window=int(best_row.get("label_forward_window", 10)),
    train_fraction=0.7,
)

best_features = generate_ma_features(best_series, best_config.ewma_short_spans, best_config.ewma_long_spans)
best_labels = generate_labels(best_series, best_config.label_forward_window, best_config.label_lookback, best_config.label_z_threshold)
best_pipeline, _ = fit_pca_momentum(best_features, best_labels, best_config)
best_signals = predict_signals(best_pipeline, best_features, smoothing=best_config.signal_smoothing)

# Pre-compute signal lookup
signal_map = best_signals.to_dict()

# Find the matching query from ALL_QUERIES
# (Use the first outright query as example — adapt tenor as needed)
bt_query = IRSwapQuery(
    structure=IRSwapStructure.OUTRIGHT,
    value=IRSwapValue.NPV,
    curve=CURVE_ID,
    tenor=SFR_OUTRIGHTS[0].tenor,
    structure_kwargs={"bpv": 100_000},
)


def signal_fetch(t):
    """Lookup pre-computed signal at timestamp t."""
    return signal_map.get(t, 0)


buy_trigger = Trigger(
    trigger_requirements=MktTriggerRequirements(
        fetch=signal_fetch, op=operator.gt, threshold=0.5
    ),
    actions=[
        UnwindPositionsAction(match_all=True),
        AddQueryAction(query=bt_query),
    ],
)

sell_query = IRSwapQuery(
    structure=IRSwapStructure.OUTRIGHT,
    value=IRSwapValue.NPV,
    curve=CURVE_ID,
    tenor=SFR_OUTRIGHTS[0].tenor,
    structure_kwargs={"bpv": -100_000},
)

sell_trigger = Trigger(
    trigger_requirements=MktTriggerRequirements(
        fetch=signal_fetch, op=operator.lt, threshold=-0.5
    ),
    actions=[
        UnwindPositionsAction(match_all=True),
        AddQueryAction(query=sell_query),
    ],
)

# Build event-driven backtest
tg = TimeGrid(ts_range)
strategy = QueryStrategy(name="PCA Momentum", triggers=[buy_trigger, sell_trigger])

bt = QueryDrivenBacktest(
    time_grid=tg,
    mdp=curve_mdp,
    strategy=strategy,
)
bt.run()

mtm = pd.Series(bt.mtm_history).sort_index()

fig, ax = plt.subplots(figsize=(16, 8))
ax.plot(mtm.index, mtm.values, linewidth=1)
ax.set_title(f"QueryDrivenBacktest: PCA Momentum on {best_contract_col}")
ax.set_ylabel("MTM ($)")
ax.axhline(0, color='black', linewidth=0.5)
plt.tight_layout()
plt.show()